# 02 - Préparation : choix de la cadence et du nombre minimum de coups
On lit les 2M premières parties (~24h du 1er sept.) en gardant seulement 180+0 et 180+2.

In [1]:
from pathlib import Path
import polars as pl
from pgn import read_games

PGN_PATH = Path("../Data/lichess_db_standard_rated_2026-09.pgn.zst")
RAW_PATH = Path("../Data/blitz_180_raw.parquet")

# Lecture longue (~1 min) -> on la fait une seule fois puis on relit le parquet
if not RAW_PATH.exists():
    read_games(PGN_PATH, n_read=2_000_000, time_controls={"180+0", "180+2"}).write_parquet(RAW_PATH)
df = pl.read_parquet(RAW_PATH)
df.shape

(557237, 22)

In [ ]:
# ici le df on peut le sauvegarder en parquet
df.head()

Event,Site,Date,Round,White,Black,Result,UTCDate,UTCTime,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,ECO,Opening,TimeControl,Termination,Movetext,WhiteTitle,BlackTitle,moves,n_moves
str,str,str,str,str,str,str,str,str,i32,i32,i32,i32,str,str,str,str,str,str,str,list[str],u32
"""Rated Blitz game""","""https://lichess.org/IMOSvErG""","""2026.09.01""","""-""","""DOMI60FR""","""Holizox""","""1/2-1/2""","""2026.09.01""","""00:00:20""",1735,1664,-2,2,"""D08""","""Queen's Gambit Declined: Albin…","""180+2""","""Normal""","""1. d4 { [%clk 0:03:00] } 1... …",null,null,"[""d4"", ""d5"", … ""Kc4""]",99
"""Rated Blitz game""","""https://lichess.org/bs18a5LO""","""2026.09.01""","""-""","""Q8KSA""","""IRS007""","""1-0""","""2026.09.01""","""00:00:22""",1991,2045,7,-7,"""C42""","""Petrov's Defense: Three Knight…","""180+0""","""Normal""","""1. e4 { [%clk 0:03:00] } 1... …",null,null,"[""e4"", ""e5"", … ""Qe6+""]",85
"""Rated Blitz game""","""https://lichess.org/YXYiz29z""","""2026.09.01""","""-""","""MiltonMorales""","""Glaidel""","""0-1""","""2026.09.01""","""00:00:23""",2224,2217,-6,6,"""B23""","""Sicilian Defense: Closed""","""180+0""","""Normal""","""1. e4 { [%clk 0:03:00] } 1... …",null,null,"[""e4"", ""c5"", … ""Nf5""]",28
"""Rated Blitz game""","""https://lichess.org/nnqUdpEC""","""2026.09.01""","""-""","""nallek""","""Botonems""","""1-0""","""2026.09.01""","""00:00:23""",1744,1649,4,-4,"""A00""","""Saragossa Opening""","""180+0""","""Time forfeit""","""1. c3 { [%clk 0:03:00] } 1... …",null,null,"[""c3"", ""e6"", … ""Bd1""]",65
"""Rated Blitz game""","""https://lichess.org/kXbkGSNm""","""2026.09.01""","""-""","""evgeniy_duk""","""Juanca_Master_Mind""","""0-1""","""2026.09.01""","""00:00:10""",1557,1611,-5,5,"""D40""","""Queen's Gambit Declined: Semi-…","""180+2""","""Normal""","""1. d4 { [%clk 0:03:00] } 1... …",null,null,"[""d4"", ""d5"", … ""Rxh4#""]",70


## 1. Quelle cadence ?
On compare le volume et la dispersion des Elo (les 2 joueurs empilés).

In [2]:
elos = df.unpivot(index="TimeControl", on=["WhiteElo", "BlackElo"], value_name="elo")

elos.group_by("TimeControl").agg(
    n_parties=pl.len() // 2,
    elo_moyen=pl.col("elo").mean().round(),
    elo_std=pl.col("elo").std().round(),
    pct_moins_1000=((pl.col("elo") < 1000).mean() * 100).round(1),
    pct_plus_2200=((pl.col("elo") > 2200).mean() * 100).round(1),
)

TimeControl,n_parties,elo_moyen,elo_std,pct_moins_1000,pct_plus_2200
str,u32,f64,f64,f64,f64
"""180+2""",178653,1624.0,365.0,4.5,5.5
"""180+0""",378584,1707.0,350.0,2.8,6.8


## 2. Combien de coups minimum ?
Comment se terminent les parties selon le nombre de coups par joueur ?
Si les parties courtes finissent surtout par abandon/déconnexion, elles ne disent rien du niveau.

In [3]:
(
    df.with_columns(coups_par_joueur=(pl.col("n_moves") // 2).clip(upper_bound=20))  # 20 = "20 et plus"
    .group_by("coups_par_joueur")
    .agg(
        n_parties=pl.len(),
        pct_abandoned=((pl.col("Termination") == "Abandoned").mean() * 100).round(1),
        pct_time_forfeit=((pl.col("Termination") == "Time forfeit").mean() * 100).round(1),
    )
    .sort("coups_par_joueur")
)

coups_par_joueur,n_parties,pct_abandoned,pct_time_forfeit
u32,u32,f64,f64
0,1836,85.6,8.9
1,1012,0.0,50.4
2,872,0.0,38.1
3,1210,0.0,23.2
4,1366,0.0,24.8
…,…,…,…
16,8942,0.0,10.4
17,9843,0.0,10.3
18,10619,0.0,10.5
